In [ ]:

import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import random


# BEFORE (default):
#   col1  col2  col3  ...  col20
#   1     2     3     ...  20
#
# AFTER (with these settings):
#   col1  col2  col3  col4  col5  col6  col7  ...  col20   ← All columns visible!
#   1     2     3     4     5     6     7     ...  20
# ============================================================================




pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")

Libraries loaded successfully!
Pandas version: 2.2.2


In [ ]:
np.random.seed(42)
random.seed(42)


num_transactions = 200
start_date = datetime(2024, 1, 1)

In [ ]:
# Generate Transaction IDs


transaction_ids = []

for i in range(1001, 1001 + num_transactions):
    txn_id = "QP" + str(i).zfill(6)
    transaction_ids.append(txn_id)

print(f"Generated {len(transaction_ids)} transaction IDs")
print(f"First 5: {transaction_ids[:5]}")
print(f"Last 5: {transaction_ids[-5:]}")

Generated 200 transaction IDs
First 5: ['QP001001', 'QP001002', 'QP001003', 'QP001004', 'QP001005']
Last 5: ['QP001196', 'QP001197', 'QP001198', 'QP001199', 'QP001200']


In [ ]:

amounts = []

for _ in range(num_transactions):
    rand = random.random()

    if rand < 0.3:
        amounts.append(random.randint(100, 500))

    elif rand < 0.7:
        amounts.append(random.randint(500, 2000))

    elif rand < 0.95:
        amounts.append(random.randint(2000, 10000))

    else:
        amounts.append(random.randint(10000, 50000))

print(f"Generated {len(amounts)} amounts")
print(f"Sample amounts: {amounts[:10]}")
print(f"Min: ₹{min(amounts)}, Max: ₹{max(amounts)}, Avg: ₹{sum(amounts)//len(amounts)}")

Generated 200 amounts
Sample amounts: [551, 4006, 477, 479, 2712, 565, 211, 408, 201, 7745]
Min: ₹116, Max: ₹46422, Avg: ₹3546


In [ ]:

dates = []

for _ in range(num_transactions):
    random_days = random.randint(0, 29)
    random_hours = random.randint(8, 22)
    random_minutes = random.randint(0, 59)

    date = start_date + timedelta(days=random_days, hours=random_hours, minutes=random_minutes)
    dates.append(date)

print(f"Generated {len(dates)} dates")
print(f"First date: {dates[0]}")
print(f"Sample dates: {dates[:3]}")

Generated 200 dates
First date: 2024-01-24 20:15:00
Sample dates: [datetime.datetime(2024, 1, 24, 20, 15), datetime.datetime(2024, 1, 9, 10, 50), datetime.datetime(2024, 1, 23, 9, 24)]


In [ ]:

payment_methods = np.random.choice(
    ['UPI', 'Card', 'NetBanking', 'Wallet'],
    size=num_transactions,
    p=[0.55, 0.25, 0.12, 0.08]
)

print(f"Payment method distribution:")
unique, counts = np.unique(payment_methods, return_counts=True)


for method, count in zip(unique, counts):
    print(f"  {method}: {count} ({count/num_transactions*100:.1f}%)")

Payment method distribution:
  Card: 42 (21.0%)
  NetBanking: 27 (13.5%)
  UPI: 116 (58.0%)
  Wallet: 15 (7.5%)


In [ ]:

merchant_categories = np.random.choice(
    ['Electronics', 'Fashion', 'Food & Grocery', 'Travel', 'Entertainment'],
    size=num_transactions,
    p=[0.25, 0.30, 0.25, 0.12, 0.08]
)

regions = np.random.choice(
    ['North', 'South', 'East', 'West', 'Central'],
    size=num_transactions,
    p=[0.25, 0.30, 0.20, 0.15, 0.10]
)

print(f"Generated {len(merchant_categories)} merchant categories")
print(f"Generated {len(regions)} regions")

Generated 200 merchant categories
Generated 200 regions


In [ ]:

statuses = ['SUCCESS'] * num_transactions

print(f"All {len(statuses)} transactions marked as SUCCESS in ledger")

All 200 transactions marked as SUCCESS in ledger


In [ ]:
internal_ledger = pd.DataFrame({
    'transaction_id': transaction_ids,
    'ledger_date': dates,
    'ledger_amount': amounts,
    'payment_method': payment_methods,
    'merchant_category': merchant_categories,
    'region': regions,
    'ledger_status': statuses
})



internal_ledger = internal_ledger.sort_values('ledger_date').reset_index(drop=True)





print(f"Internal Ledger: {len(internal_ledger)} transactions")
print(f"Date range: {internal_ledger['ledger_date'].min()} to {internal_ledger['ledger_date'].max()}")
print(f"Total value: ₹{internal_ledger['ledger_amount'].sum():,}")
print()
internal_ledger.head(10)

Internal Ledger: 200 transactions
Date range: 2024-01-01 08:15:00 to 2024-01-30 20:34:00
Total value: ₹709,343



,transaction_id,ledger_date,ledger_amount,payment_method,merchant_category,region,ledger_status
0,QP001054,2024-01-01 08:15:00,783,NetBanking,Fashion,Central,SUCCESS
1,QP001175,2024-01-01 09:18:00,604,Card,Electronics,West,SUCCESS
2,QP001163,2024-01-01 15:54:00,378,Card,Fashion,North,SUCCESS
3,QP001169,2024-01-01 16:12:00,154,UPI,Electronics,Central,SUCCESS
4,QP001125,2024-01-01 17:09:00,10967,UPI,Entertainment,West,SUCCESS
5,QP001199,2024-01-01 17:18:00,5372,NetBanking,Food & Grocery,North,SUCCESS
6,QP001094,2024-01-01 21:56:00,5268,Card,Fashion,North,SUCCESS
7,QP001134,2024-01-01 22:19:00,1496,UPI,Electronics,East,SUCCESS
8,QP001195,2024-01-02 08:35:00,9997,UPI,Fashion,West,SUCCESS
9,QP001091,2024-01-02 11:53:00,933,UPI,Electronics,South,SUCCESS


In [ ]:

gateway_data = internal_ledger.copy()

gateway_data.columns = ['transaction_id', 'gateway_date', 'gateway_amount',
                        'payment_method', 'merchant_category', 'region', 'gateway_status']


print(f"Starting gateway data with {len(gateway_data)} transactions (copy of ledger)")

Starting gateway data with 200 transactions (copy of ledger)


In [ ]:
missing_count = int(len(gateway_data) * 0.05)


missing_indices = np.random.choice(gateway_data.index, size=missing_count, replace=False)

gateway_data = gateway_data.drop(missing_indices)


print(f"DISCREPANCY 1: Removed {missing_count} transactions from gateway")
print(f"Gateway now has {len(gateway_data)} transactions")

DISCREPANCY 1: Removed 10 transactions from gateway
Gateway now has 190 transactions


In [ ]:
extra_count = int(num_transactions * 0.03)

extra_transactions = []

for i in range(extra_count):
    extra_txn = {
        'transaction_id': f"QP{99000 + i}",
        'gateway_date': start_date + timedelta(days=random.randint(0, 29), hours=random.randint(8, 22)),
        'gateway_amount': random.randint(500, 5000),
        'payment_method': random.choice(['UPI', 'Card']),
        'merchant_category': random.choice(['Electronics', 'Fashion']),
        'region': random.choice(['North', 'South']),
        'gateway_status': 'SUCCESS'
    }
    extra_transactions.append(extra_txn)



extra_df = pd.DataFrame(extra_transactions)
gateway_data = pd.concat([gateway_data, extra_df], ignore_index=True)



print(f"DISCREPANCY 2: Added {extra_count} extra transactions to gateway")
print(f"These IDs exist in gateway but NOT in ledger: {[t['transaction_id'] for t in extra_transactions]}")

DISCREPANCY 2: Added 6 extra transactions to gateway
These IDs exist in gateway but NOT in ledger: ['QP99000', 'QP99001', 'QP99002', 'QP99003', 'QP99004', 'QP99005']


In [ ]:
mismatch_count = int(len(gateway_data) * 0.03)


mismatch_indices = np.random.choice(gateway_data.index, size=mismatch_count, replace=False)

for idx in mismatch_indices:
    original = gateway_data.loc[idx, 'gateway_amount']
    fee = random.choice([10, 20, 50, 100, 200])
    gateway_data.loc[idx, 'gateway_amount'] = original - fee

print(f"DISCREPANCY 3: Modified {mismatch_count} transaction amounts (fee deductions)")

DISCREPANCY 3: Modified 5 transaction amounts (fee deductions)


In [ ]:

status_mismatch_count = int(len(gateway_data) * 0.02)

status_indices = np.random.choice(gateway_data.index[:-extra_count], size=status_mismatch_count, replace=False)

for idx in status_indices:
    gateway_data.loc[idx, 'gateway_status'] = 'FAILED'

print(f"DISCREPANCY 4: Changed {status_mismatch_count} statuses to FAILED")

DISCREPANCY 4: Changed 3 statuses to FAILED


In [ ]:
# ============================================================================
# FINAL GATEWAY DATA SUMMARY
# ============================================================================
# reset_index(drop=True) renumbers rows from 0 after all our modifications
# ============================================================================

gateway_data = gateway_data.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"GATEWAY DATA SUMMARY")
print(f"{'='*50}")
print(f"Total transactions: {len(gateway_data)}")
print(f"Total value: ₹{gateway_data['gateway_amount'].sum():,}")
print(f"\nStatus distribution:")
print(gateway_data['gateway_status'].value_counts())
print()
gateway_data.head(10)


GATEWAY DATA SUMMARY
Total transactions: 196
Total value: ₹589,992

Status distribution:
gateway_status
SUCCESS    193
FAILED       3
Name: count, dtype: int64



,transaction_id,gateway_date,gateway_amount,payment_method,merchant_category,region,gateway_status
0,QP001054,2024-01-01 08:15:00,783,NetBanking,Fashion,Central,SUCCESS
1,QP001175,2024-01-01 09:18:00,604,Card,Electronics,West,SUCCESS
2,QP001163,2024-01-01 15:54:00,378,Card,Fashion,North,SUCCESS
3,QP001169,2024-01-01 16:12:00,154,UPI,Electronics,Central,SUCCESS
4,QP001125,2024-01-01 17:09:00,10967,UPI,Entertainment,West,SUCCESS
5,QP001199,2024-01-01 17:18:00,5372,NetBanking,Food & Grocery,North,SUCCESS
6,QP001094,2024-01-01 21:56:00,5268,Card,Fashion,North,SUCCESS
7,QP001134,2024-01-01 22:19:00,1496,UPI,Electronics,East,SUCCESS
8,QP001195,2024-01-02 08:35:00,9997,UPI,Fashion,West,SUCCESS
9,QP001091,2024-01-02 11:53:00,933,UPI,Electronics,South,SUCCESS


In [ ]:

print(f"\n1. RECORD COUNTS")
print(f"   Internal Ledger: {len(internal_ledger)} transactions")
print(f"   Gateway Data: {len(gateway_data)} transactions")
print(f"   Difference: {len(internal_ledger) - len(gateway_data)}")

print(f"\n2. DATE RANGES")
print(f"   Ledger: {internal_ledger['ledger_date'].min().date()} to {internal_ledger['ledger_date'].max().date()}")
print(f"   Gateway: {gateway_data['gateway_date'].min().date()} to {gateway_data['gateway_date'].max().date()}")

print(f"\n3. TOTAL AMOUNTS")
print(f"   Ledger Total: ₹{internal_ledger['ledger_amount'].sum():,}")
print(f"   Gateway Total: ₹{gateway_data['gateway_amount'].sum():,}")
print(f"   Difference: ₹{internal_ledger['ledger_amount'].sum() - gateway_data['gateway_amount'].sum():,}")

print(f"\n4. NULL VALUES")
print(f"   Ledger nulls: {internal_ledger.isnull().sum().sum()}")
print(f"   Gateway nulls: {gateway_data.isnull().sum().sum()}")



1. RECORD COUNTS
   Internal Ledger: 200 transactions
   Gateway Data: 196 transactions
   Difference: 4

2. DATE RANGES
   Ledger: 2024-01-01 to 2024-01-30
   Gateway: 2024-01-01 to 2024-01-30

3. TOTAL AMOUNTS
   Ledger Total: ₹709,343
   Gateway Total: ₹589,992
   Difference: ₹119,351

4. NULL VALUES
   Ledger nulls: 0
   Gateway nulls: 0


In [ ]:

ledger_ids = set(internal_ledger['transaction_id'])
gateway_ids = set(gateway_data['transaction_id'])

print("TRANSACTION SET ANALYSIS")
print("-" * 40)
print(f"Total unique IDs in Ledger: {len(ledger_ids)}")
print(f"Total unique IDs in Gateway: {len(gateway_ids)}")

only_in_ledger = ledger_ids - gateway_ids
only_in_gateway = gateway_ids - ledger_ids
in_both = ledger_ids & gateway_ids

print(f"\nOnly in Ledger (missing from gateway): {len(only_in_ledger)}")
print(f"Only in Gateway (missing from ledger): {len(only_in_gateway)}")
print(f"Present in both systems: {len(in_both)}")

TRANSACTION SET ANALYSIS
----------------------------------------
Total unique IDs in Ledger: 200
Total unique IDs in Gateway: 196

Only in Ledger (missing from gateway): 10
Only in Gateway (missing from ledger): 6
Present in both systems: 190


In [ ]:
# Step 2: Extract Missing Transactions


only_in_ledger_list = list(only_in_ledger)


missing_in_gateway = internal_ledger[internal_ledger['transaction_id'].isin(only_in_ledger_list)].copy()


missing_in_gateway['issue_type'] = 'MISSING_IN_GATEWAY'
missing_in_gateway['issue_description'] = 'Transaction recorded in ledger but not found in gateway'

print(f"MISSING IN GATEWAY: {len(missing_in_gateway)} transactions")
print(f"Value at risk: ₹{missing_in_gateway['ledger_amount'].sum():,}")
print()
missing_in_gateway[['transaction_id', 'ledger_amount', 'payment_method', 'issue_type']].head()


MISSING IN GATEWAY: 10 transactions
Value at risk: ₹134,928



,transaction_id,ledger_amount,payment_method,issue_type
28,QP001165,44664,UPI,MISSING_IN_GATEWAY
79,QP001092,8187,Card,MISSING_IN_GATEWAY
116,QP001017,32059,UPI,MISSING_IN_GATEWAY
124,QP001030,5113,UPI,MISSING_IN_GATEWAY
135,QP001045,6646,UPI,MISSING_IN_GATEWAY


In [ ]:

only_in_gateway_list = list(only_in_gateway)

missing_in_ledger = gateway_data[gateway_data['transaction_id'].isin(only_in_gateway_list)].copy()

missing_in_ledger['issue_type'] = 'MISSING_IN_LEDGER'
missing_in_ledger['issue_description'] = 'Transaction found in gateway but not recorded in ledger'

print(f"MISSING IN LEDGER: {len(missing_in_ledger)} transactions")
print(f"Unrecorded value: ₹{missing_in_ledger['gateway_amount'].sum():,}")
print()
missing_in_ledger[['transaction_id', 'gateway_amount', 'payment_method', 'issue_type']].head()

MISSING IN LEDGER: 6 transactions
Unrecorded value: ₹15,847



,transaction_id,gateway_amount,payment_method,issue_type
190,QP99000,920,Card,MISSING_IN_LEDGER
191,QP99001,3782,Card,MISSING_IN_LEDGER
192,QP99002,2989,UPI,MISSING_IN_LEDGER
193,QP99003,3909,UPI,MISSING_IN_LEDGER
194,QP99004,2935,Card,MISSING_IN_LEDGER


In [ ]:
# Find Amount and Status Mismatches

in_both_list = list(in_both)

ledger_common = internal_ledger[internal_ledger['transaction_id'].isin(in_both_list)].copy()
gateway_common = gateway_data[gateway_data['transaction_id'].isin(in_both_list)].copy()

comparison = ledger_common.merge(
    gateway_common[['transaction_id', 'gateway_amount', 'gateway_status']],
    on='transaction_id',
    how='inner'
)

print(f"Comparing {len(comparison)} transactions present in both systems...")
comparison.head()

Comparing 190 transactions present in both systems...


,transaction_id,ledger_date,ledger_amount,payment_method,merchant_category,region,ledger_status,gateway_amount,gateway_status
0,QP001054,2024-01-01 08:15:00,783,NetBanking,Fashion,Central,SUCCESS,783,SUCCESS
1,QP001175,2024-01-01 09:18:00,604,Card,Electronics,West,SUCCESS,604,SUCCESS
2,QP001163,2024-01-01 15:54:00,378,Card,Fashion,North,SUCCESS,378,SUCCESS
3,QP001169,2024-01-01 16:12:00,154,UPI,Electronics,Central,SUCCESS,154,SUCCESS
4,QP001125,2024-01-01 17:09:00,10967,UPI,Entertainment,West,SUCCESS,10967,SUCCESS


In [ ]:

comparison['amount_diff'] = comparison['ledger_amount'] - comparison['gateway_amount']

comparison['status_match'] = comparison['ledger_status'] == comparison['gateway_status']

amount_mismatches = comparison[comparison['amount_diff'] != 0].copy()
amount_mismatches['issue_type'] = 'AMOUNT_MISMATCH'
amount_mismatches['issue_description'] = 'Amount differs between ledger and gateway'

print(f"\nAMOUNT MISMATCHES: {len(amount_mismatches)} transactions")
print(f"Total discrepancy: ₹{amount_mismatches['amount_diff'].sum():,}")
print()
amount_mismatches[['transaction_id', 'ledger_amount', 'gateway_amount', 'amount_diff']].head()


AMOUNT MISMATCHES: 5 transactions
Total discrepancy: ₹270



,transaction_id,ledger_amount,gateway_amount,amount_diff
48,QP001129,3348,3328,20
60,QP001181,1375,1175,200
70,QP001114,1993,1973,20
100,QP001081,8813,8793,20
150,QP001140,1587,1577,10


In [ ]:

status_mismatches = comparison[comparison['status_match'] == False].copy()
status_mismatches['issue_type'] = 'STATUS_MISMATCH'
status_mismatches['issue_description'] = 'Status differs: Ledger says SUCCESS, Gateway says FAILED'

print(f"STATUS MISMATCHES: {len(status_mismatches)} transactions")
print(f"Value affected: ₹{status_mismatches['ledger_amount'].sum():,}")
print()
status_mismatches[['transaction_id', 'ledger_amount', 'ledger_status', 'gateway_status']].head()

STATUS MISMATCHES: 3 transactions
Value affected: ₹3,218



,transaction_id,ledger_amount,ledger_status,gateway_status
14,QP001111,693,SUCCESS,FAILED
49,QP001117,472,SUCCESS,FAILED
126,QP001013,2053,SUCCESS,FAILED


In [ ]:

discrepancy_report = []

# Add missing in gateway (HIGH severity — potential revenue loss)
for idx, row in missing_in_gateway.iterrows():
    discrepancy_report.append({
        'transaction_id': row['transaction_id'],
        'date': row['ledger_date'],
        'ledger_amount': row['ledger_amount'],
        'gateway_amount': 0,
        'difference': row['ledger_amount'],
        'payment_method': row['payment_method'],
        'issue_type': 'MISSING_IN_GATEWAY',
        'severity': 'HIGH',
        'action_required': 'Investigate with gateway provider'
    })

# Add missing in ledger (MEDIUM severity — unrecorded revenue)
for idx, row in missing_in_ledger.iterrows():
    discrepancy_report.append({
        'transaction_id': row['transaction_id'],
        'date': row['gateway_date'],
        'ledger_amount': 0,
        'gateway_amount': row['gateway_amount'],
        'difference': -row['gateway_amount'],
        'payment_method': row['payment_method'],
        'issue_type': 'MISSING_IN_LEDGER',
        'severity': 'MEDIUM',
        'action_required': 'Add to ledger or investigate source'
    })

# Add amount mismatches (severity depends on amount)
for idx, row in amount_mismatches.iterrows():
    severity = 'MEDIUM' if abs(row['amount_diff']) < 100 else 'HIGH'
    discrepancy_report.append({
        'transaction_id': row['transaction_id'],
        'date': row['ledger_date'],
        'ledger_amount': row['ledger_amount'],
        'gateway_amount': row['gateway_amount'],
        'difference': row['amount_diff'],
        'payment_method': row['payment_method'],
        'issue_type': 'AMOUNT_MISMATCH',
        'severity': severity,
        'action_required': 'Verify fee deductions or partial refunds'
    })

# Add status mismatches (CRITICAL severity — potential loss!)
for idx, row in status_mismatches.iterrows():
    discrepancy_report.append({
        'transaction_id': row['transaction_id'],
        'date': row['ledger_date'],
        'ledger_amount': row['ledger_amount'],
        'gateway_amount': row['gateway_amount'],
        'difference': row['ledger_amount'],
        'payment_method': row['payment_method'],
        'issue_type': 'STATUS_MISMATCH',
        'severity': 'CRITICAL',
        'action_required': 'URGENT: Payment may have failed'
    })



# Convert to DataFrame and sort by difference (biggest problems first)
discrepancies_df = pd.DataFrame(discrepancy_report)
discrepancies_df = discrepancies_df.sort_values('difference', ascending=False).reset_index(drop=True)

print(f"CONSOLIDATED DISCREPANCY REPORT")
print(f"Total issues: {len(discrepancies_df)}")
print(f"\nBy issue type:")
print(discrepancies_df['issue_type'].value_counts())
print(f"\nBy severity:")
print(discrepancies_df['severity'].value_counts())
print()
discrepancies_df.head(10)

CONSOLIDATED DISCREPANCY REPORT
Total issues: 24

By issue type:
issue_type
MISSING_IN_GATEWAY    10
MISSING_IN_LEDGER      6
AMOUNT_MISMATCH        5
STATUS_MISMATCH        3
Name: count, dtype: int64

By severity:
severity
HIGH        11
MEDIUM      10
CRITICAL     3
Name: count, dtype: int64



,transaction_id,date,ledger_amount,gateway_amount,difference,payment_method,issue_type,severity,action_required
0,QP001165,2024-01-07 13:51:00,44664,0,44664,UPI,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
1,QP001017,2024-01-19 12:02:00,32059,0,32059,UPI,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
2,QP001029,2024-01-22 22:21:00,25256,0,25256,Card,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
3,QP001056,2024-01-25 09:29:00,9054,0,9054,Wallet,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
4,QP001092,2024-01-14 14:57:00,8187,0,8187,Card,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
5,QP001045,2024-01-22 11:32:00,6646,0,6646,UPI,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
6,QP001030,2024-01-20 13:42:00,5113,0,5113,UPI,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
7,QP001013,2024-01-21 16:25:00,2053,2053,2053,NetBanking,STATUS_MISMATCH,CRITICAL,URGENT: Payment may have failed
8,QP001026,2024-01-22 19:47:00,1942,0,1942,Card,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider
9,QP001012,2024-01-25 12:22:00,1069,0,1069,Wallet,MISSING_IN_GATEWAY,HIGH,Investigate with gateway provider


In [ ]:

# Find perfectly matched transactions (no issues at all)
matched_transactions = comparison[
    (comparison['amount_diff'] == 0) &
    (comparison['status_match'] == True)
].copy()



# Calculate key business metrics
total_gmv = internal_ledger['ledger_amount'].sum()
confirmed_gmv = matched_transactions['ledger_amount'].sum()
at_risk_amount = discrepancies_df['difference'].sum()


success_count = len(matched_transactions)
total_count = len(internal_ledger)
success_rate = success_count / total_count * 100

avg_ticket = internal_ledger['ledger_amount'].mean()



# Group by dimensions for breakdown analysis
payment_method_breakdown = internal_ledger.groupby('payment_method').agg({
    'transaction_id': 'count',
    'ledger_amount': 'sum'
}).reset_index()
payment_method_breakdown.columns = ['payment_method', 'transaction_count', 'total_amount']



category_breakdown = internal_ledger.groupby('merchant_category').agg({
    'transaction_id': 'count',
    'ledger_amount': 'sum'
}).reset_index()
category_breakdown.columns = ['merchant_category', 'transaction_count', 'total_amount']



region_breakdown = internal_ledger.groupby('region').agg({
    'transaction_id': 'count',
    'ledger_amount': 'sum'
}).reset_index()
region_breakdown.columns = ['region', 'transaction_count', 'total_amount']




# Daily trend for time series chart
daily_trend = internal_ledger.copy()
daily_trend['date'] = daily_trend['ledger_date'].dt.date
daily_summary = daily_trend.groupby('date').agg({
    'transaction_id': 'count',
    'ledger_amount': 'sum'
}).reset_index()
daily_summary.columns = ['date', 'transaction_count', 'total_amount']

print("\nKEY BUSINESS METRICS")
print("-" * 40)
print(f"Total GMV (Ledger): ₹{total_gmv:,.2f}")
print(f"Confirmed GMV: ₹{confirmed_gmv:,.2f}")
print(f"Amount at Risk: ₹{at_risk_amount:,.2f}")
print(f"Reconciliation Rate: {success_rate:.1f}%")
print(f"Average Ticket Size: ₹{avg_ticket:,.2f}")
print(f"Total Transactions: {total_count}")
print(f"Discrepancies Found: {len(discrepancies_df)}")


KEY BUSINESS METRICS
----------------------------------------
Total GMV (Ledger): ₹709,343.00
Confirmed GMV: ₹554,081.00
Amount at Risk: ₹122,569.00
Reconciliation Rate: 91.0%
Average Ticket Size: ₹3,546.72
Total Transactions: 200
Discrepancies Found: 24


In [ ]:

summary_metrics = pd.DataFrame([{
    'report_date': datetime.now().strftime('%Y-%m-%d'),
    'total_gmv': total_gmv,
    'confirmed_gmv': confirmed_gmv,
    'at_risk_amount': at_risk_amount,
    'total_transactions': total_count,
    'matched_transactions': success_count,
    'reconciliation_rate': success_rate,
    'avg_ticket_size': avg_ticket,
    'total_discrepancies': len(discrepancies_df),
    'critical_issues': len(discrepancies_df[discrepancies_df['severity'] == 'CRITICAL']),
    'high_issues': len(discrepancies_df[discrepancies_df['severity'] == 'HIGH']),
    'medium_issues': len(discrepancies_df[discrepancies_df['severity'] == 'MEDIUM'])
}])

print("SUMMARY METRICS (for dashboard scorecards)")
print(summary_metrics.T)

SUMMARY METRICS (for dashboard scorecards)
                               0
report_date           2026-04-19
total_gmv                 709343
confirmed_gmv             554081
at_risk_amount            122569
total_transactions           200
matched_transactions         182
reconciliation_rate         91.0
avg_ticket_size         3546.715
total_discrepancies           24
critical_issues                3
high_issues                   11
medium_issues                 10


In [ ]:
# Export all DataFrames to CSV
summary_metrics.to_csv('summary_metrics.csv', index=False)
daily_summary.to_csv('daily_summary.csv', index=False)
payment_method_breakdown.to_csv('payment_method_breakdown.csv', index=False)
category_breakdown.to_csv('category_breakdown.csv', index=False)
region_breakdown.to_csv('region_breakdown.csv', index=False)
discrepancies_df.to_csv('discrepancies.csv', index=False)

print("All files exported successfully!")
print("\\nFiles created:")
print("- summary_metrics.csv")
print("- daily_summary.csv")
print("- payment_method_breakdown.csv")
print("- category_breakdown.csv")
print("- region_breakdown.csv")
print("- discrepancies.csv")

All files exported successfully!
\nFiles created:
- summary_metrics.csv
- daily_summary.csv
- payment_method_breakdown.csv
- category_breakdown.csv
- region_breakdown.csv
- discrepancies.csv
